Part 1: Data Ingestion & Storage

In [25]:
import requests
import pandas as pd
import polars as pl
import duckdb
import time

try:
    # 1. Progommatic Donwload of the data and 3.File Organization
    def download_file(url):
        filename = "data/raw/" + url.split('/')[-1]
        with requests.get(url, stream=True) as reference:
            reference.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in reference.iter_content(chunk_size=8192):
                    f.write(chunk)
        return filename

    parquetfile = download_file('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet')
    csvfile = download_file('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')

    lazy_parquetfile = pl.scan_parquet(parquetfile)
    lazy_csvfile = pl.scan_csv(csvfile)
    
    combined_lazy_df = lazy_parquetfile.join(lazy_csvfile, left_on="PULocationID", right_on="LocationID", how="left")

    df_polars = combined_lazy_df.collect()

    # 2. Data Validation
    required_columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    ]

    missing_columns = [col for col in required_columns if col not in df_polars.columns]

    if not missing_columns:
        print("All required columns are present.")
    else:    
        print("Some required columns are missing.")
        exit(1)
    
    for col in df_polars.columns:
        if "datetime" in col:
            if df_polars[col].dtype != pl.Datetime:
                print(f"Column '{col}' is not in datetime format.")
                exit(1)
    
    print("All datetime columns are in the correct format.")
    
    print(f"Row count: {df_polars.height}")
    
except Exception as e:    
    print(f"An error occurred Validating the data: {e}")
    exit(1)


All required columns are present.
All datetime columns are in the correct format.
Row count: 2964624


Part 2: Data Transformation & Analysis

In [26]:
rows = df_polars.height
filtered_df = df_polars.filter(
    (pl.col("tpep_pickup_datetime").is_not_null()) &
    (pl.col("tpep_dropoff_datetime").is_not_null()) &
    (pl.col("PULocationID").is_not_null()) &
    (pl.col("DOLocationID").is_not_null()) &
    (pl.col("fare_amount").is_not_null())
).select(pl.all())

filtered = rows - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to null values: {filtered}")
else:
    print(f"No rows removed due to null values.")

filtered_df = df_polars.filter( 
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") >= 0 ) &
    (pl.col("fare_amount") >= 500) 
).select(pl.all())

filtered = filtered - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to invalid trips: {filtered}")
else:
    print(f"No rows removed due to invalid trips.")

filtered_df = filtered_df.filter(
    (pl.col("tpep_pickup_datetime") < pl.col("tpep_dropoff_datetime"))
    ).select(pl.all())

filtered = filtered - filtered_df.height
if filtered > 0:
    print(f"Rows removed due to pickup datetime being after dropoff datetime: {filtered}")
else:
    print("No rows removed due to pickup datetime being after dropoff datetime.")
    
    
engineered = filtered_df.with_columns([
# Calculate trip duration in minutes
((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
.dt.total_seconds() / 60).alias('trip_duration_minutes'),

# Calculate trip speed in miles per hour
(pl.col('trip_distance') / (pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime')).dt.total_seconds()).alias('trip_speed_mph'),

# Extract hour of day
pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),

# Extract day of week
pl.col('tpep_pickup_datetime').alias('pickup_day_of_weekday')   
])





No rows removed due to null values.
No rows removed due to invalid trips.
No rows removed due to pickup datetime being after dropoff datetime.


In [35]:
print(engineered.schema)
con = duckdb.connect()
busy_pickup = con.execute('''
    SELECT
    COUNT(*) AS trip_count, zone
    FROM engineered
    GROUP BY zone
    ORDER BY trip_count DESC
    LIMIT 10
    ''').fetchdf()
print(busy_pickup)

Schema({'VendorID': Int32, 'tpep_pickup_datetime': Datetime(time_unit='ns', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='ns', time_zone=None), 'passenger_count': Int64, 'trip_distance': Float64, 'RatecodeID': Int64, 'store_and_fwd_flag': String, 'PULocationID': Int32, 'DOLocationID': Int32, 'payment_type': Int64, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64, 'Borough': String, 'Zone': String, 'service_zone': String, 'trip_duration_minutes': Float64, 'trip_speed_mph': Float64, 'pickup_hour': Int8, 'pickup_day_of_weekday': Datetime(time_unit='ns', time_zone=None)})
   trip_count                        Zone
0          20                 JFK Airport
1           3           LaGuardia Airport
2           2      Mott Haven/Port Morris
3           2              Outside of NYC
4           1         L

In [43]:
avg_fare_by_hour = con.execute('''
    SELECT
        pickup_hour,
    AVG(fare_amount) AS avg_fare
    FROM engineered
    GROUP BY pickup_hour
    ORDER BY pickup_hour
    ''').fetchdf()
print(avg_fare_by_hour)

    pickup_hour     avg_fare
0             0   550.550000
1             7  1616.500000
2             8   579.800000
3             9   504.250000
4            10   898.440000
5            11   500.000000
6            12   625.200000
7            13   530.800000
8            14   607.800000
9            15   670.450000
10           16   899.000000
11           17   631.366667
12           18   602.900000
13           19   659.500000
14           21   693.080000
15           23   744.300000


In [47]:
payment_type_distribution = con.execute('''
    SELECT
        payment_type,
        COUNT(*) AS trip_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM engineered
    GROUP BY payment_type
    ORDER BY percentage DESC
    ''').fetchdf()
print(payment_type_distribution)

   payment_type  trip_count  percentage
0             2          26       76.47
1             4           3        8.82
2             3           3        8.82
3             1           2        5.88


In [70]:
tip_percentage_by_dow = con.execute('''
    SELECT
        DAYNAME(pickup_day_of_weekday) AS day_of_week,
        AVG(CAST(tip_amount AS DECIMAL) / CAST(fare_amount AS DECIMAL) * 100.0) AS avg_tip_percentage
    FROM engineered
    WHERE payment_type = 1
    GROUP BY DAYOFWEEK(pickup_day_of_weekday), day_of_week
    ORDER BY DAYOFWEEK(pickup_day_of_weekday)
    ''').fetchdf()
print(tip_percentage_by_dow)

  day_of_week  avg_tip_percentage
0     Tuesday                 0.0
1   Wednesday                 0.0


In [ ]:
top_zone_pairs = con.execute('''
    SELECT
        PULocation.Zone AS pickup_zone,
        DOLocation.Zone AS dropoff_zone,
    COUNT(*) AS trip_count
    FROM engineered AS trips
    LEFT JOIN lazy_csvfile AS PULocation ON trips.PULocationID = PULocation.LocationID
    LEFT JOIN lazy_csvfile AS DOLocation ON trips.DOLocationID = DOLocation.LocationID
    GROUP BY pickup_zone, dropoff_zone
    ORDER BY trip_count DESC
    LIMIT 5
    ''').fetchdf()
print(top_zone_pairs)

                  pickup_zone                dropoff_zone  trip_count
0                 JFK Airport              Outside of NYC          18
1           LaGuardia Airport              Outside of NYC           3
2      Mott Haven/Port Morris              Outside of NYC           2
3              Outside of NYC              Outside of NYC           2
4  Spuyten Duyvil/Kingsbridge  Spuyten Duyvil/Kingsbridge           1
